In [95]:
!pip install -q langchain langchain-groq langchain-community wikipedia tavily-python

In [96]:
import os
from getpass import getpass
from google.colab import userdata

os.environ["GROQ_API_KEY"] = getpass()
os.environ["TAVILY_API_KEY"] = getpass("Tavily API Key: ")

··········
Tavily API Key: ··········


In [113]:
from langchain_groq import ChatGroq
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain.tools import tool
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
import math

In [114]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0
)

In [144]:
@tool

def calculator(expression : str) -> str:
  """
    Useful for solving math calculations.
    Input must be a valid Python math expression.
    Examples: '2 + 2', 'math.sqrt(144)', '15 * 4 / 2'
  """
  try:
    result = eval(expression,{'math':math,'__builtins__':{}})
    return f"[TOOL USED: calculator]\n\n{result}"

  except Exception as e:
      return f"Error: {str(e)}"

In [145]:
@tool
def wikipedia_search(topic: str) -> str:
    """
    Search Wikipedia for factual information about a person,
    place, historical event, scientific concept, technology,
    or other general topic.

    Input should be a topic such as 'Albert Einstein',
    'Machine Learning', or 'Daniel Radcliffe'.
    """

    wikipedia = WikipediaQueryRun(
        api_wrapper=WikipediaAPIWrapper(
            top_k_results=2,
            doc_content_chars_max=4000
        )
    )

    try:
        result = wikipedia.invoke(topic)

        return f"[TOOL USED: wikipedia_search]\n{result}"

    except Exception as e:
        return f"[TOOL USED: wikipedia_search]\nWikipedia search error: {str(e)}"

In [146]:
@tool
def web_search(query: str) -> str:
    """
    Search the live web for current or recent information.

    Use this when the user asks about recent news,
    current events, latest information, or information
    that may not be available in the local knowledge base.

    Input should be a natural-language search query.
    """

    tavily = TavilySearchResults(
        max_results=3
    )

    try:
        result = tavily.invoke(query)
        return f"[TOOL USED: web_search]\n\n{result}"

    except Exception as e:
        return f"[TOOL USED: web_search]\nWeb search failed: {str(e)}"

In [147]:
tools = [calculator,wikipedia_search,web_search]

In [148]:
for i in tools:
  print(i.name)


calculator
wikipedia_search
web_search


In [149]:
react_prompt = PromptTemplate.from_template("""You are a helpful AI assistant. Answer the question using the tools available.

You have access to these tools:
{tools}

Use EXACTLY this format:

Question: the input question
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: input to the action
Observation: result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: the final answer to the question

Important rules:
- Always start with Thought:
- Action must be exactly one of the tool names listed
- Always end with Final Answer:

Begin!

Question: {input}
Thought:{agent_scratchpad}""")


In [150]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_prompt,)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=6,
    handle_parsing_errors=True)

print('we are ready')

we are ready


In [151]:
r = agent_executor.invoke({"input":"What is 56-10?"})
print('answer :-',r['output'])



> Entering new AgentExecutor chain...
Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Question: What is 56-10?
Thought: I need to compute the subtraction 56 minus 10. I will use the calculator tool.
Action: calculator
Action Input: 56 - 10[TOOL USED: calculator]

46Thought: I have computed the subtraction using the calculator tool and obtained the result.
Final Answer: 46

> Finished chain.
answer :- 46


In [152]:
r = agent_executor.invoke({
    "input": "tell me about daniel radcliff? use Wikipedia"
})

print("🤖:", r["output"])



> Entering new AgentExecutor chain...
Question: tell me about daniel radcliff? use Wikipedia  
Thought: I need to retrieve information about Daniel Radcliffe from Wikipedia.  
Action: wikipedia_search  
Action Input: Daniel Radcliffe  [TOOL USED: wikipedia_search]
Page: Daniel Radcliffe
Summary: Daniel Jacob Radcliffe (born 23 July 1989) is an English actor best known for portraying Harry Potter in all eight films of the Harry Potter film series from 2001 to 2011. Radcliffe's films as a leading actor have grossed over $8.38 billion worldwide. His accolades include a Critics' Choice Award and a Tony Award, in addition to nominations for a BAFTA Award, two Grammy Awards, and a Primetime Emmy Award. He was featured on the Forbes 30 Under 30 Europe list of 2019.
Radcliffe branched out to stage acting in 2007, starring in the West End and Broadway productions of Equus. He returned to Broadway in the musical How to Succeed in Business Without Really Trying (2011), earning a Grammy Award no

In [153]:
r = agent_executor.invoke({
    "input": "Search the web for the latest developments in artificial intelligence."
})

print("🤖:", r["output"])



> Entering new AgentExecutor chain...
Question: Search the web for the latest developments in artificial intelligence.
Thought: I need to retrieve up-to-date information about recent AI developments, so I will use the web_search tool.
Action: web_search
Action Input: latest developments in artificial intelligence 2024[TOOL USED: web_search]

[{'title': "Year in review: Google's biggest AI advancements of 2024", 'url': 'https://blog.google/innovation-and-ai/products/2024-ai-extraordinary-progress-advancement', 'content': 'At the start of 2024, we introduced ImageFX, a new generative AI tool that creates images from text prompts, and MusicFX, a tool for creating up-to-70-second audio clips also based on text prompts. At I/O, we shared an early preview of MusicFX DJ, a tool that helps bring the joy of live music creation to more people. In October, we collaborated with Jacob Collier on making MusicFX DJ simpler to use, especially for new or aspiring musicians. And we updated our music A